In [76]:
import pandas as pd
import os 

In [77]:
usuario = os.getlogin() 

In [78]:
base = pd.read_excel(fr"C:\Users\{usuario}\Downloads\nuevo-f\formulario-ang\bases\UnidadesIMB_CS!_v2.xlsx",sheet_name="Sheet 1")
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")

In [79]:
import pandas as pd

sheet_id = "1maRNGDuU9rEFWZLgMdhJS1waAnJxl6ENntm-nyD0tq8"
gid = "1765182479"

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

base_an = pd.read_csv(url)



In [80]:
col = ["clues_imb"]
base = base[col]

In [81]:
base = base.merge(
    clues[["clues_imb", "entidad"]],
    on="clues_imb",
    how="left"
)

In [82]:
colum =['clues_imb', 'entidad','consultorio','pregunta']
base_an = base_an[colum]    

In [83]:
b = pd.read_excel(fr"C:\Users\{usuario}\Downloads\nuevo-f\formulario-ang\bases\UM_IMB_SUS.xlsx",sheet_name="Hoja2")

In [84]:
b = b.drop(index=[0, 2, 5,3,1])

In [85]:
b = b.drop(columns=['Unnamed: 1'])

In [86]:
b = b.rename(columns={
   'CLUES' : 'preguntas',
})

In [87]:
b.columns

Index(['preguntas'], dtype='object')

In [88]:
base

,clues_imb,entidad
0,BCIMB000051,BAJA CALIFORNIA
1,BCIMB000092,BAJA CALIFORNIA
2,BCIMB000162,BAJA CALIFORNIA
3,BCIMB000186,BAJA CALIFORNIA
4,BCIMB000191,BAJA CALIFORNIA
...,...,...
2822,GRIMB010244,GUERRERO
2823,GRIMB010256,GUERRERO
2824,GRIMB010565,GUERRERO
2825,SRIMB001071,SONORA


In [89]:
base_an.head()

,clues_imb,entidad,consultorio,pregunta
0,BSIMB000025,BAJA CALIFORNIA SUR,NaN,internet
1,BSIMB000025,BAJA CALIFORNIA SUR,1.0,turno_consultorio_1
2,BCIMB000092,BAJA CALIFORNIA,NaN,internet
3,BSIMB000136,BAJA CALIFORNIA SUR,NaN,internet
4,BSIMB000136,BAJA CALIFORNIA SUR,1.0,turno_consultorio_1


In [90]:
base_an = base_an[
    ~base_an["pregunta"].str.contains("internet|turno_consultorio", case=False, na=False)
]

In [91]:
base_conteo =base["clues_imb"].unique()
base_conteo

array(['BCIMB000051', 'BCIMB000092', 'BCIMB000162', ..., 'GRIMB010565',
       'SRIMB001071', 'TCIMB000704'], shape=(2191,), dtype=object)

In [92]:
total_unidades = base["clues_imb"].nunique()

respondieron = base_an["clues_imb"].nunique()

sin_responder = total_unidades - respondieron

avance_general = round(respondieron / total_unidades * 100, 1)

## tablas

In [93]:
# TABLA 1 - AVANCE POR ENTIDAD

# Unidades esperadas
esperadas = (
    base
    .groupby("entidad")
    .size()
    .reset_index(name="esperadas")
)

# Unidades que respondieron
contestadas = (
    base_an[["entidad","clues_imb"]]
    .drop_duplicates()
    .groupby("entidad")
    .size()
    .reset_index(name="respondieron")
)

tabla_entidades = (
    esperadas
    .merge(contestadas,
           on="entidad",
           how="left")
)

tabla_entidades["respondieron"] = (
    tabla_entidades["respondieron"]
    .fillna(0)
    .astype(int)
)

tabla_entidades["porcentaje"] = (
    tabla_entidades["respondieron"]
    / tabla_entidades["esperadas"]
    *100
).round(1)

tabla_entidades = tabla_entidades.sort_values(
    "porcentaje",
    ascending=False
)


In [94]:
tabla_entidades

,entidad,esperadas,respondieron,porcentaje
15,SAN LUIS POTOSI,33,4,12.1
14,QUINTANA ROO,165,15,9.1
0,BAJA CALIFORNIA,92,4,4.3
21,ZACATECAS,23,1,4.3
3,CHIAPAS,25,1,4.0
20,VERACRUZ DE IGNACIO DE LA LLAVE,128,5,3.9
9,MICHOACAN DE OCAMPO,43,1,2.3
2,CAMPECHE,45,1,2.2
1,BAJA CALIFORNIA SUR,70,1,1.4
19,TAMAULIPAS,210,1,0.5


In [95]:
# TABLA 2 - COMPLETITUD POR UNIDAD

# Número de preguntas del formulario
n_preguntas = len(b)

# Número de consultorios por unidad
consultorios = (
    base_an
    .groupby(["clues_imb","entidad"],as_index=False)
    ["consultorio"]
    .max()
)

consultorios.rename(
    columns={"consultorio":"consultorios"},
    inplace=True
)

# Preguntas respondidas
respondidas = (
    base_an
    .groupby(["clues_imb","entidad"])
    .size()
    .reset_index(name="respondidas")
)

tabla_unidades = consultorios.merge(
    respondidas,
    on=["clues_imb","entidad"]
)

tabla_unidades["esperadas"] = (
    tabla_unidades["consultorios"]
    * n_preguntas
)

tabla_unidades["porcentaje"] = (
    tabla_unidades["respondidas"]
    / tabla_unidades["esperadas"]
    *100
).round(1)

tabla_unidades = (
    tabla_unidades
    .rename(columns={"clues_imb":"clues"})
    .sort_values("porcentaje")
)

In [96]:
tabla_unidades

,clues,entidad,consultorios,respondidas,esperadas,porcentaje
5,CCIMB000102,CAMPECHE,1.0,1,44.0,2.3
19,QRIMB001536,QUINTANA ROO,1.0,1,44.0,2.3
7,MNIMB001316,MICHOACAN DE OCAMPO,1.0,2,44.0,4.5
0,BCIMB000092,BAJA CALIFORNIA,2.0,4,88.0,4.5
27,TSIMB001646,TAMAULIPAS,1.0,2,44.0,4.5
4,BSIMB000136,BAJA CALIFORNIA SUR,1.0,4,44.0,9.1
6,CSIMB003651,CHIAPAS,1.0,5,44.0,11.4
13,QRIMB000906,QUINTANA ROO,1.0,26,44.0,59.1
24,SPIMB000701,SAN LUIS POTOSI,1.0,36,44.0,81.8
20,QRIMB001652,QUINTANA ROO,2.0,87,88.0,98.9


In [97]:
# LISTAS PARA JINJA2
entidades = tabla_entidades.to_dict(orient="records")
unidades = tabla_unidades.to_dict(orient="records")

In [98]:
tabla_entidades = (
    tabla_unidades
    .groupby("entidad", as_index=False)
    .agg(
        consultorios=("consultorios","sum"),
        respondidas=("respondidas","sum"),
        esperadas=("esperadas","sum"),
        unidades=("clues","count")
    )
)

tabla_entidades["porcentaje"] = (
    tabla_entidades["respondidas"]
    / tabla_entidades["esperadas"]
    *100
).round(1)

tabla_entidades = tabla_entidades.sort_values(
    "porcentaje",
    ascending=False
)

In [99]:
tabla_entidades

,entidad,consultorios,respondidas,esperadas,unidades,porcentaje
8,VERACRUZ DE IGNACIO DE LA LLAVE,10.0,440,440.0,5,100.0
9,ZACATECAS,6.0,262,264.0,1,99.2
6,SAN LUIS POTOSI,5.0,212,220.0,4,96.4
5,QUINTANA ROO,23.0,950,1012.0,15,93.9
0,BAJA CALIFORNIA,5.0,136,220.0,4,61.8
3,CHIAPAS,1.0,5,44.0,1,11.4
1,BAJA CALIFORNIA SUR,1.0,4,44.0,1,9.1
4,MICHOACAN DE OCAMPO,1.0,2,44.0,1,4.5
7,TAMAULIPAS,1.0,2,44.0,1,4.5
2,CAMPECHE,1.0,1,44.0,1,2.3


In [100]:
# Total de unidades por entidad (base completa)
total_por_entidad = (
    base
    .groupby("entidad")["clues_imb"]
    .nunique()
    .reset_index(name="total_unidades")
)

# Unidades que han respondido al menos una pregunta
respondieron_por_entidad = (
    base_an[["entidad", "clues_imb"]]
    .drop_duplicates()
    .groupby("entidad")["clues_imb"]
    .nunique()
    .reset_index(name="unidades_respondieron")
)

tabla_avance = total_por_entidad.merge(
    respondieron_por_entidad,
    on="entidad",
    how="left"
)

tabla_avance["unidades_respondieron"] = (
    tabla_avance["unidades_respondieron"].fillna(0).astype(int)
)

tabla_avance["porcentaje"] = (
    tabla_avance["unidades_respondieron"]
    / tabla_avance["total_unidades"]
    * 100
).round(1)

tabla_avance = tabla_avance.sort_values("porcentaje", ascending=False)

In [101]:
tabla_avance


,entidad,total_unidades,unidades_respondieron,porcentaje
15,SAN LUIS POTOSI,28,4,14.3
14,QUINTANA ROO,133,15,11.3
20,VERACRUZ DE IGNACIO DE LA LLAVE,74,5,6.8
0,BAJA CALIFORNIA,73,4,5.5
21,ZACATECAS,19,1,5.3
3,CHIAPAS,24,1,4.2
9,MICHOACAN DE OCAMPO,27,1,3.7
2,CAMPECHE,31,1,3.2
1,BAJA CALIFORNIA SUR,51,1,2.0
19,TAMAULIPAS,150,1,0.7


In [102]:
import plotly.graph_objects as go

fig = go.Figure(
    go.Pie(
        labels=tabla_avance["entidad"],
        values=tabla_avance["porcentaje"],   # ← ahora el tamaño depende del porcentaje
        hole=0.6,
        pull=[0.03] * len(tabla_avance),
        texttemplate="%{label}<br>%{value:.1f}%",
        textinfo="none",
        hovertemplate="<b>%{label}</b><br>"
                      "Avance: %{value:.1f}%<extra></extra>"
    )
)

fig.update_layout(
    title="Avance por entidad",
    width=900,
    height=700,
    legend_title="Entidad",
    template="plotly_white"
)

fig.show()

In [103]:
from pathlib import Path
import json

base_dir = Path.cwd()
salida_data = base_dir / "reporte_data.js"
html_origen = base_dir / "reporte_interactivo.html"
index_destino = base_dir / "index.html"

payload = {
    "tabla_avance": tabla_avance.to_dict(orient="records"),
    "tabla_entidades": tabla_entidades.to_dict(orient="records"),
    "tabla_unidades": tabla_unidades.to_dict(orient="records"),
}

contenido_js = "window.REPORTE_DATA = " + json.dumps(payload, ensure_ascii=False, indent=2) + ";\n"
salida_data.write_text(contenido_js, encoding="utf-8")

if html_origen.exists():
    index_destino.write_text(html_origen.read_text(encoding="utf-8"), encoding="utf-8")
    print(f"Index exportado en: {index_destino}")
else:
    print(f"No se encontro {html_origen.name}; index.html no se actualizo.")

print(f"Datos exportados en: {salida_data}")
print("Abre index.html en la misma carpeta para ver el reporte.")

No se encontro reporte_interactivo.html; index.html no se actualizo.
Datos exportados en: c:\Users\jose.valdez\Downloads\graficas\reporte_data.js
Abre index.html en la misma carpeta para ver el reporte.


In [104]:

import plotly.graph_objects as go

# Ordenar de mayor a menor porcentaje para cascada (mayor arriba)
df_plot = tabla_entidades.sort_values('porcentaje', ascending=True)

colores = ['#2ecc71' if p == 100 else '#3498db' if p >= 80 else '#e67e22' if p >= 50 else '#e74c3c'
           for p in df_plot['porcentaje']]

fig_cascada = go.Figure()

fig_cascada.add_trace(go.Bar(
    x=df_plot['porcentaje'],
    y=df_plot['entidad'],
    orientation='h',
    marker=dict(color=colores),
    text=[f"{p:.1f}%" for p in df_plot['porcentaje']],
    textposition='outside',
    textfont=dict(size=11),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Porcentaje: %{x:.1f}%<br>"
        "<extra></extra>"
    )
))

fig_cascada.update_layout(
    title=dict(
        text="<b>Gráfica de llenado de consultorios</b>",
        x=0.5,
        font=dict(size=16)
    ),
    xaxis=dict(
        title="Porcentaje (%)",
        range=[0, 115],
        ticksuffix="%",
        showgrid=True,
        gridcolor='lightgrey'
    ),
    yaxis=dict(
        title="Entidad",
        automargin=True
    ),
    plot_bgcolor='white',
    height=500,
    margin=dict(l=20, r=60, t=60, b=40)
)

fig_cascada.show()
